# Structured Output With Pydantic

In [46]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field


class Person(BaseModel):
    """
    人物信息
    """
    name: str = Field(description="Person's name")
    age: int = Field(description="Person's age")
    is_xnn: bool = Field(description="是否是小男娘")


load_dotenv(verbose=True, override=True)
base_url = os.getenv("OPENROUTER_API_BASE")
api_key = os.getenv("OPENROUTER_API_KEY")

model = init_chat_model(model="openrouter:openai/gpt-5.6-sol")
model_with_structured_output = model.with_structured_output(Person, method="json_schema")

response = model_with_structured_output.invoke("给出 驹泽乃依 的人物信息")
print(response)
print(type(response))

name='驹泽乃依' age=-1 is_xnn=False
<class '__main__.Person'>


In [13]:
from pydantic import BaseModel, Field


class MovieModel(BaseModel):
    """
    电影的详细信息
    """
    title: str = Field(description="电影标题")
    year: int = Field(description="电影上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="电影评分，满分十分")


model_with_structure = model.with_structured_output(MovieModel)
response = model_with_structure.invoke("给出盗梦空间的信息")

print(response)
print(type(response))

title='盗梦空间' year=2010 director='克里斯托弗·诺兰' rating=8.8
<class '__main__.MovieModel'>


In [18]:
from pydantic import BaseModel, Field


# 定义输出结构
class SentimentAnalysis(BaseModel):
    """情感分析结果"""
    sentiment: str = Field(description="情感倾向：positive/negative/neutral")
    confidence: float = Field(description="置信度，0-1之间")
    keywords: list[str] = Field(description="关键词列表")


model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

# ✅ v1.x：使用 with_structured_output
structured_model = model.with_structured_output(SentimentAnalysis)

# 调用
text = "这个课程内容很实用，学到了很多知识，强烈推荐！"
result = structured_model.invoke(
    f"分析以下文本的情感：\n{text}"
)

print(f"类型: {type(result)}")  # <class 'SentimentAnalysis'>
print(f"情感: {result.sentiment}")
print(f"置信度: {result.confidence}")
print(f"关键词: {result.keywords}")

类型: <class '__main__.SentimentAnalysis'>
情感: positive
置信度: 0.95
关键词: ['实用', '学到了很多知识', '强烈推荐']


## 高级特性

### Optional

In [25]:
from typing import Optional

model = init_chat_model(model="openrouter:openai/gpt-5.6-luna")


class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: Optional[int] = Field(description="年龄")
    occupation: Optional[str] = Field(description="职业")


model_with_structured_output = model.with_structured_output(Person)
response = model_with_structured_output.invoke("给出 驹泽乃依 的人物信息")
print(response)

name='驹泽乃依' age=None occupation=None


### Default

In [26]:
from typing import Optional

model = init_chat_model(model="openrouter:openai/gpt-5.6-luna")


class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(18, description="年龄")
    occupation: Optional[str] = Field(description="职业")


model_with_structured_output = model.with_structured_output(Person)
response = model_with_structured_output.invoke("给出 驹泽乃依 的人物信息")
print(response)

name='驹泽乃依' age=18 occupation=None


In [27]:


model = init_chat_model(model="openrouter:openai/gpt-5.6-luna")


class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(18, description="年龄")
    occupation: str | None = Field(description="职业")


model_with_structured_output = model.with_structured_output(Person)
response = model_with_structured_output.invoke("给出 驹泽乃依 的人物信息")
print(response)

name='驹泽乃依' age=18 occupation=None


### 枚举类型

In [28]:
from enum import StrEnum


class Priority(StrEnum):
    LOW = "低"
    MEDIUM = "中"
    HIGH = "高"


class Status(StrEnum):
    ACTIVE = "激活"
    INACTIVE = "未激活"


class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Priority = Field(description="紧急程度")


# 测试
structured_llm = model.with_structured_output(CustomerInfo)
conversation = """
客服: 您好，请问有什么可以帮助您？
客户: 我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服: 好的，我帮您查一下
"""

result = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")
print(result)

name='王小明' phone='138-1234-5678' email=None issue='订单一直没发货' urgency=<Priority.HIGH: '高'>


In [30]:
from typing import Optional, Literal


class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Literal["LOW", "MEDIUM", "HIGH"] = Field(description="紧急程度")


# 测试
structured_llm = model.with_structured_output(CustomerInfo)
conversation = """
客服: 您好，请问有什么可以帮助您？
客户: 我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服: 好的，我帮您查一下
"""

result = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")
print(result)

name='王小明' phone='138-1234-5678' email=None issue='订单一直没发货' urgency='HIGH'


### 列表提取

In [31]:
class Person(BaseModel):
    """人物信息"""
    name: str
    age: int


class PersonList(BaseModel):
    people: list[Person] = Field(description="多个 Person 对象")


structured_output = model.with_structured_output(PersonList)
response = structured_output.invoke("张三 30岁，李四 25岁")

print(response)

people=[Person(name='张三', age=30), Person(name='李四', age=25)]


In [32]:
class Review(BaseModel):
    """产品评论"""
    product: str
    rating: int = Field(description="评分 1-5")
    pros: list[str] = Field(description="优点列表")
    cons: list[str] = Field(description="缺点列表")


structured_llm = model.with_structured_output(Review)
review = structured_llm.invoke("""
iPhone 17 很棒！摄像头强大，手感好。但是价格贵，没有充电器。4分。
""")

print(review)

product='iPhone 17' rating=4 pros=['摄像头强大', '手感好'] cons=['价格贵', '不附带充电器']


In [33]:
class Invoice(BaseModel):
    """发票信息"""
    invoice_number: str = Field(description="发票号")
    date: str = Field(description="日期")
    total_amount: float = Field(description="总金额")
    items: list[str] = Field(description="商品")


# 测试
structured_llm = model.with_structured_output(Invoice)
invoice_text = """
发票号: INV-2024-001
日期: 2024-01-15
总金额: 1299.00
商品: MacBook Pro, AppleCare+
"""

invoice = structured_llm.invoke(f"提取发票信息：{invoice_text}")
print(invoice)

invoice_number='INV-2024-001' date='2024-01-15' total_amount=1299.0 items=['MacBook Pro', 'AppleCare+']


### 嵌套结构

In [34]:
class Address(BaseModel):
    """地点描述"""
    city: str
    district: str


class Company(BaseModel):
    """公司信息"""
    name: str
    address: Address  # 嵌套模型


structured_llm = model.with_structured_output(Company)
result = structured_llm.invoke("阿里巴巴在杭州滨江区")
print(result)

name='阿里巴巴' address=Address(city='杭州', district='滨江区')


In [35]:
from pydantic import BaseModel, Field


# 1. 定义嵌套的 Pydantic 模型
class Actor(BaseModel):
    """演员信息"""
    name: str = Field(description="演员姓名")
    role: str = Field(description="饰演的角色")


class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")
    director: str = Field(description="导演")
    cast: list[Actor] = Field(description="演员列表")  # 定义列表字段
    rating: float = Field(description="评分")


# 2. 初始化模型并绑定输出结构
structured_model = model.with_structured_output(Movie)

# 3. 调用模型，直接获取 Movie 实例
response = structured_model.invoke("请介绍电影《盗梦空间》")

# 4. 访问嵌套数据
print(f"电影名: {response.title}")
print(f"上映年份: {response.year}")
print(f"导演: {response.director}")
print(f"演员列表: {response.cast}")
print(f"评分: {response.rating}")

电影名: 盗梦空间
上映年份: 2010
导演: 克里斯托弗·诺兰
演员列表: [Actor(name='莱昂纳多·迪卡普里奥', role='多姆·科布'), Actor(name='约瑟夫·高登-莱维特', role='亚瑟'), Actor(name='艾伦·佩吉', role='阿丽雅德妮'), Actor(name='汤姆·哈迪', role='伊姆斯'), Actor(name='渡边谦', role='斋藤'), Actor(name='玛丽昂·歌迪亚', role='梅尔')]
评分: 9.3


In [36]:
from pydantic import BaseModel


class Aspect(BaseModel):
    """评论维度"""
    name: str = Field(description="维度名称，如：质量、价格、服务")
    score: int = Field(description="评分，1-5")
    comment: str = Field(description="具体评价")


class ProductReview(BaseModel):
    """产品评论分析"""
    overall_sentiment: Literal["positive", "negative", "neutral"] = Field(description="整体情感")
    overall_score: int = Field(description="综合评分，1-5")
    aspects: list[Aspect] = Field(description="各维度评价")
    summary: str = Field(description="一句话总结")


# 创建结构化模型
structured_model = model.with_structured_output(ProductReview)

# 测试
review_text = """
这款笔记本电脑性能非常强大，运行大型软件毫无压力。
屏幕色彩鲜艳，看视频很舒服。
不过价格有点贵，而且风扇噪音较大。
客服态度很好，物流也快。
总体来说还是值得购买的。
"""

result = structured_model.invoke(
    f"分析以下产品评论：\n{review_text}"
)

print(f"整体情感: {result.overall_sentiment}")
print(f"综合评分: {result.overall_score}/5")
print(f"\n各维度评价:")

for aspect in result.aspects:
    print(f" - {aspect.name}: {aspect.score}/5 - {aspect.comment}")

print(f"\n总结: {result.summary}")

整体情感: positive
综合评分: 4/5

各维度评价:
 - 性能: 5/5 - 性能非常强大，运行大型软件毫无压力。
 - 屏幕: 5/5 - 屏幕色彩鲜艳，观看视频体验舒适。
 - 价格: 3/5 - 价格有点贵，性价比方面略有不足。
 - 噪音: 2/5 - 风扇噪音较大，可能影响使用体验。
 - 客服: 5/5 - 客服态度很好，服务令人满意。
 - 物流: 5/5 - 物流速度快，配送体验良好。

总结: 这款笔记本性能和屏幕表现出色，客服与物流服务良好，但价格偏高且风扇噪音较大，总体仍值得购买。


### 限制条件

In [37]:
from pydantic import ValidationError


class User(BaseModel):
    name: str = Field(min_length=2, max_length=20)
    age: int = Field(ge=0, le=150)
    email: str


print("\n有效数据:")
try:
    user = User(name="张三", age=30, email="zhang@example.com")
    print(f"[OK] {user.name}, {user.age}, {user.email}")
except ValidationError as e:
    print(f"[FAIL] {e}")
    print("\n无效数据（年龄超出范围）:")

try:
    user = User(name="李四", age=200, email="li@example.com")
    print(f"[OK] {user}")
except ValidationError as e:
    print(f"[FAIL] 验证失败（符合预期）: {e.errors()[0]['msg']}")


有效数据:
[OK] 张三, 30, zhang@example.com
[FAIL] 验证失败（符合预期）: Input should be less than or equal to 150


In [42]:
from langchain_core.messages import SystemMessage, HumanMessage


class Product(BaseModel):
    """产品信息（严格验证）"""
    name: str = Field(description="产品名称（字符串类型）", min_length=2)
    price: float = Field(description="价格，数字类型", gt=0)
    stock: int = Field(description="库存，整数类型", ge=0)


model = init_chat_model(model="openrouter:openai/gpt-5.6-sol-pro")

# 测试
structured_llm = model.with_structured_output(Product)

# response = structured_llm.invoke("华为mate 80 promax 价格是7999，当前库存 100")
messages = [SystemMessage("根据用户输入如实结构化输出，不要编造虚假值"),
            HumanMessage("华为mate 80 promax 价格是-7999，当前库 存-100")]
response = structured_llm.invoke(messages)
print(response)

name='华为mate 80 promax' price=7999.0 stock=100
